# Custom Dataset Tutorial: Train on Your Own Histopathology Data

This notebook shows how to adapt HistoCore to your own whole slide images (WSI) and patch-level datasets.

**What you'll learn:**
- Prepare your custom dataset
- Create a custom DataLoader
- Configure the model for your task
- Train and evaluate

**Supported formats:** TIFF, SVS, NDPI, and more via OpenSlide

## 1. Dataset Structure

Organize your data in this structure:

```
data/my_dataset/
├── train/
│   ├── class_0/
│   │   ├── patch_001.png
│   │   ├── patch_002.png
│   │   └── ...
│   └── class_1/
│       ├── patch_001.png
│       └── ...
├── val/
│   ├── class_0/
│   └── class_1/
└── test/
    ├── class_0/
    └── class_1/
```

## 2. Create Custom Dataset Class

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
import numpy as np

class CustomHistoDataset(Dataset):
    """Custom dataset for histopathology patches."""
    
    def __init__(self, root_dir, split='train', transform=None):
        """
        Args:
            root_dir: Path to dataset root
            split: 'train', 'val', or 'test'
            transform: Optional transforms to apply
        """
        self.root_dir = Path(root_dir) / split
        self.transform = transform
        
        # Find all image files
        self.samples = []
        self.classes = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        for class_name in self.classes:
            class_dir = self.root_dir / class_name
            for img_path in class_dir.glob('*.png'):
                self.samples.append((img_path, self.class_to_idx[class_name]))
        
        print(f"Loaded {len(self.samples)} samples from {split} set")
        print(f"Classes: {self.classes}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        # Load image
        image = Image.open(img_path).convert('RGB')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.float32)

# Example usage
print("Custom dataset class created!")

## 3. Define Data Transforms

Standard augmentations for histopathology:

In [ ]:
# Training transforms (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((96, 96)),  # Adjust to your patch size
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/test transforms (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Transforms defined!")

## 4. Create DataLoaders

In [ ]:
# Configuration
data_root = 'data/my_dataset'  # Change to your dataset path
batch_size = 256
num_workers = 4  # Adjust based on your CPU

# Create datasets
train_dataset = CustomHistoDataset(data_root, split='train', transform=train_transform)
val_dataset = CustomHistoDataset(data_root, split='val', transform=val_transform)
test_dataset = CustomHistoDataset(data_root, split='test', transform=val_transform)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True if num_workers > 0 else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True if num_workers > 0 else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

print(f"\nDataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

## 5. Configure Model for Your Task

In [ ]:
from src.models.attention_mil import AttentionMIL

# Model configuration
num_classes = len(train_dataset.classes)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = AttentionMIL(
    feature_extractor='resnet18',  # or 'resnet50', 'efficientnet_b0'
    pretrained=True,
    feature_dim=512,
    hidden_dim=256,
    num_classes=num_classes,
    dropout=0.3
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel created:")
print(f"  Architecture: AttentionMIL with ResNet18")
print(f"  Number of classes: {num_classes}")
print(f"  Total parameters: {total_params:,}")
print(f"  Device: {device}")

## 6. Training Setup

In [ ]:
import torch.optim as optim
from torch.cuda.amp import GradScaler

# Loss function
if num_classes == 2:
    criterion = torch.nn.BCEWithLogitsLoss()
else:
    criterion = torch.nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.01
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=15,  # number of epochs
    eta_min=1e-6
)

# Mixed precision training
scaler = GradScaler()

print("Training setup complete!")

## 7. Training Loop

In [ ]:
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score

def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision forward pass
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        # Backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            
            # Collect predictions
            if num_classes == 2:
                preds = torch.sigmoid(outputs)
            else:
                preds = torch.softmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    if num_classes == 2:
        auc = roc_auc_score(all_labels, all_preds)
        acc = accuracy_score(all_labels, (all_preds > 0.5).astype(int))
    else:
        auc = roc_auc_score(all_labels, all_preds, multi_class='ovr')
        acc = accuracy_score(all_labels, all_preds.argmax(axis=1))
    
    return {
        'loss': total_loss / len(loader),
        'auc': auc,
        'accuracy': acc
    }

print("Training functions defined!")

## 8. Train the Model

In [ ]:
num_epochs = 15
best_val_auc = 0

print(f"Training for {num_epochs} epochs...\n")

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
    
    # Validate
    val_metrics = validate(model, val_loader, criterion, device)
    
    # Update learning rate
    scheduler.step()
    
    # Print metrics
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_metrics['loss']:.4f}")
    print(f"  Val AUC: {val_metrics['auc']:.4f}")
    print(f"  Val Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"  LR: {scheduler.get_last_lr()[0]:.6f}")
    
    # Save best model
    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  ✓ Saved best model (AUC: {best_val_auc:.4f})")
    
    print()

print(f"Training complete! Best validation AUC: {best_val_auc:.4f}")

## 9. Final Evaluation on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pth'))

# Evaluate on test set
test_metrics = validate(model, test_loader, criterion, device)

print("\n" + "="*50)
print("FINAL TEST SET RESULTS")
print("="*50)
print(f"Test Loss: {test_metrics['loss']:.4f}")
print(f"Test AUC: {test_metrics['auc']:.4f}")
print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
print("="*50)

## 10. Save Model for Deployment

In [ ]:
# Save complete model with metadata
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'test_auc': test_metrics['auc'],
    'test_accuracy': test_metrics['accuracy'],
    'num_classes': num_classes,
    'classes': train_dataset.classes,
    'config': {
        'feature_extractor': 'resnet18',
        'hidden_dim': 256,
        'dropout': 0.3
    }
}, 'final_model.pth')

print("\n✅ Model saved to 'final_model.pth'")
print("\nYou can now deploy this model for inference!")

## Next Steps

1. **Hyperparameter Tuning**: Experiment with learning rates, batch sizes, architectures
2. **Model Interpretability**: Use Grad-CAM to visualize what the model learned
3. **Clinical Deployment**: Integrate with PACS systems
4. **Federated Learning**: Train across multiple hospitals with privacy
5. **Continuous Monitoring**: Track model performance in production

See the [documentation](https://matthewvaishnav.github.io/histocore/) for more advanced features!